In [0]:
bronze_df = spark.table("plstocks.bronze_cashflow")

In [0]:
# bronze_df.display()

In [0]:
from pyspark.sql.functions import substring_index, regexp_replace, concat_ws, to_date, col, current_timestamp, lit

silver_df = bronze_df \
    .na.drop(how="any") \
    .dropDuplicates() \
    .withColumnRenamed('year', 'date_pl') \
    .withColumn("month", substring_index("date_pl", " ", 1)) \
    .withColumn("year", substring_index("date_pl", " ", -1)) \
    .withColumn("month", regexp_replace("month", "sty", "01")) \
    .withColumn("month", regexp_replace("month", "lut", "02")) \
    .withColumn("month", regexp_replace("month", "mar", "03")) \
    .withColumn("month", regexp_replace("month", "kwi", "04")) \
    .withColumn("month", regexp_replace("month", "maj", "05")) \
    .withColumn("month", regexp_replace("month", "cze", "06")) \
    .withColumn("month", regexp_replace("month", "lip", "07")) \
    .withColumn("month", regexp_replace("month", "sie", "08")) \
    .withColumn("month", regexp_replace("month", "wrz", "09")) \
    .withColumn("month", regexp_replace("month", "paź", "10")) \
    .withColumn("month", regexp_replace("month", "lis", "11")) \
    .withColumn("month", regexp_replace("month", "gru", "12")) \
    .withColumn("date", concat_ws(" ", "month", "year")) \
    .withColumn("date", to_date(col('date'), 'MM yy')) \
    .withColumn('insert_timestamp', current_timestamp()) \
    .withColumn("month", col("month").cast('int')) \
    .withColumn("year", col("year").cast('int')+2000) \
    .withColumn("free_cashflow", col("free_cashflow").cast('int')*1000)

# silver_df.display()

In [0]:
from delta.tables import DeltaTable

table_name = 'plstocks.silver_cashflow'
if spark.catalog.tableExists(table_name):
    ExistingCashflowTable = DeltaTable.forName(spark, table_name)
    ExistingCashflowTable.alias("existing") \
        .merge(
            silver_df.alias("updates"),
            "existing.ticker = updates.ticker AND existing.date = updates.date AND existing.free_cashflow != updates.free_cashflow"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)